# Kaggle GPU setup — paraphrase detection project

Before running: **Settings → Accelerator → GPU (T4/P100)** and **Settings → Internet → On**.

Then just Run All. The tunnel cell runs in the background, so Run All does not
get stuck; its sign-in code appears in the cell after it (re-run that cell if
the log is still empty).

In [ ]:
!nvidia-smi

## VS Code tunnel (name: `kaggle-gpu`)

Downloads the VS Code CLI and starts a tunnel **in the background** so the
remaining cells still run. The next cell prints the device-login URL + code —
open the URL, sign in (GitHub account), and the tunnel goes live. Then connect
from local VS Code: `Remote-Tunnels: Connect to Tunnel…` → `kaggle-gpu`.

In [ ]:
import subprocess, os

os.chdir("/kaggle/working")

# Download and extract the VS Code CLI (only if not already done)
if not os.path.exists("code"):
    subprocess.run(
        "curl -Lsk 'https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64' --output vscode_cli.tar.gz",
        shell=True, check=True
    )
    subprocess.run("tar -xf vscode_cli.tar.gz", shell=True, check=True)

# Start the tunnel as a true background process (Popen, not shell &)
log_file = open("/kaggle/working/tunnel.log", "w")
tunnel_process = subprocess.Popen(
    ["./code", "tunnel", "--accept-server-license-terms", "--name", "kaggle-gpu"],
    stdout=log_file, stderr=subprocess.STDOUT, cwd="/kaggle/working"
)

print(f"Tunnel starting in background (PID {tunnel_process.pid}) — run the next cell for the sign-in code")

In [ ]:
import time
time.sleep(10)
print(open('/kaggle/working/tunnel.log').read())

## Clone the project

In [ ]:
REPO_URL = "https://github.com/PRANAVTANGUTURU123/paraphrase-detection.git"

REPO_DIR = "/kaggle/working/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## Environment

Fresh venv + requirements. Torch's CUDA wheel is large — this cell takes a few
minutes on Kaggle's connection.

In [ ]:
!python -m venv .venv
!.venv/bin/pip install -r requirements.txt --break-system-packages -q
!.venv/bin/pip list | grep -Ei 'torch|sentence|transformers|datasets|accelerate'

## Final check — CUDA visible from the project venv

If this prints the GPU name and a successful matmul, you are ready. Full
training (from the VS Code tunnel terminal or a new cell):

```bash
.venv/bin/python -m src.train --compare                        # bi-encoder pair, full data
.venv/bin/python -m src.train --datasets qqp paws --model-type cross --tag "Cross-encoder QQP+PAWS"
```

(QQP has ~364k pairs — consider `--sample_size 50000` for a first full-GPU pass.)

In [ ]:
!.venv/bin/python check_gpu.py